In [4]:
import pyvisa

rm = pyvisa.ResourceManager()
resources = rm.list_resources()
print("Available resources:", resources)


Available resources: ('USB0::0x1313::0x80C8::M01254445::INSTR', 'ASRL1::INSTR', 'ASRL2::INSTR', 'ASRL10::INSTR')


In [4]:
import pyvisa

rm = pyvisa.ResourceManager()
dc2200 = rm.open_resource('USB0::0x1313::0x80C8::M01254445::INSTR')

# Optional: set a timeout and chunk size
dc2200.timeout = 2000  # in milliseconds
dc2200.chunk_size = 1024

# Query identity
idn = dc2200.query("*IDN?")
print("Device ID:", idn)


Device ID: Thorlabs,DC2200,M01254445,1.5.2



In [2]:
import pyvisa
import time

# --- Setup
rm = pyvisa.ResourceManager()
dc2200 = rm.open_resource('USB0::0x1313::0x80C8::M01254445::INSTR')
dc2200.timeout = 2000

# --- Optional: safety check
print("Connected to:", dc2200.query("*IDN?").strip())

# --- Parameters
on_time = 0.5  # seconds
off_time = 0.5  # seconds
total_duration = 30  # seconds

# --- Set desired current before starting
dc2200.write("SOUR:CURR 0.2")  # 200 mA, adjust as needed

# --- Loop
start_time = time.time()
print("Starting LED blinking...")

while time.time() - start_time < total_duration:
    dc2200.write("OUTP ON")
    time.sleep(on_time)
    dc2200.write("OUTP OFF")
    time.sleep(off_time)

print("Done.")


Connected to: Thorlabs,DC2200,M01254445,1.5.2
Starting LED blinking...


KeyboardInterrupt: 

In [9]:
import pyvisa
import time
import numpy as np

rm = pyvisa.ResourceManager()
dc2200 = rm.open_resource('USB0::0x1313::0x80C8::M01254445::INSTR')

# Parameters
min_current = 0.05   # in A (50 mA)
max_current = 0.35   # in A (350 mA)
steps = 20
delay = 0.2  # seconds between steps

# Generate ramp sequence
ramp_up = np.linspace(min_current, max_current, steps)
ramp_down = np.linspace(max_current, min_current, steps)
sequence = np.concatenate((ramp_up, ramp_down))

# Turn on LED
dc2200.write("OUTP ON")

# Ramp current
for curr in sequence:
    dc2200.write(f"SOUR:CURR {curr:.4f}")
    print(f"Set current: {curr*1000:.1f} mA")
    time.sleep(delay)

# Turn off LED
dc2200.write("OUTP OFF")
print("Done ramping intensity.")


VisaIOError: VI_ERROR_SYSTEM_ERROR (-1073807360): Unknown system error (miscellaneous error).

Final test: spelling "FUCK OFF" with the LED using morse code

In [8]:
import pyvisa
import time

# Morse Code Map
MORSE_CODE = {
    'F': '..-.',
    'U': '..-',
    'C': '-.-.',
    'K': '-.-',
    'O': '---'
}

# Message to spell
MESSAGE = "FUCK OFF"

# Timing (in seconds)
UNIT = 0.2
DOT = UNIT
DASH = 3 * UNIT
INTRA_CHAR = UNIT       # between dots/dashes
INTER_CHAR = 3 * UNIT   # between letters
INTER_WORD = 7 * UNIT   # between words

# LED settings
CURRENT = 0.8  # A

# Connect to DC2200
rm = pyvisa.ResourceManager()
dc2200 = rm.open_resource('USB0::0x1313::0x80C8::M01254445::INSTR')
dc2200.timeout = 2000
dc2200.write(f"SOUR:CURR {CURRENT}")
dc2200.write("OUTP OFF")

def flash(duration):
    dc2200.write("OUTP ON")
    time.sleep(duration)
    dc2200.write("OUTP OFF")

def send_letter(letter):
    code = MORSE_CODE.get(letter.upper(), '')
    for i, symbol in enumerate(code):
        if symbol == '.':
            flash(DOT)
        elif symbol == '-':
            flash(DASH)
        if i < len(code) - 1:
            time.sleep(INTRA_CHAR)

def send_message(message):
    for i, char in enumerate(message):
        if char == ' ':
            time.sleep(INTER_WORD)
        else:
            send_letter(char)
            if i < len(message) - 1 and message[i+1] != ' ':
                time.sleep(INTER_CHAR)

# Fire it up
print("Sending Morse Code: FUCK OFF")
send_message(MESSAGE)
print("Done.")


Sending Morse Code: FUCK OFF
Done.
